**请介绍一下数据仓库的分层架构，以及你在项目中的实践**
  
在我之前的项目中，我们采用的是标准的五层架构：ODS、DIM、DWD、DWS和ADS。
简单来说，ODS层就是贴源层，源系统的数据原封不动地同步过来，不做太多处理。这样做的好处是，如果源系统数据结构变了，我们只需要改ODS的同步逻辑，不会影响下游。
DIM层是维度层，统一管理用户、商品、时间这些维度数据。
DWD层是我最关注的，这是明细事实层。我们有个很重要的原则——一个业务过程一张表。比如下单是一张表，支付是另一张表，不会混在一起。而且我们规定一个指标只能归属一张表，不允许同一指标在多张表里重复定义。之前遇到过这个问题，"支付金额"这个字段在两张表里都有，但命名不一样，一个叫payment_amt，一个叫pay_amount，下游用的时候经常搞混，所以需要在命名上区分它们
DWS层是汇总层，也是指标的统一出口。所有跨业务过程的指标组装都在这一层完成。比如说"用户下单后24小时内的支付率"，这个指标涉及下单和支付两个过程，就必须在DWS层做，DWD层不碰这种跨过程的指标。
ADS层就是看业务需求来灵活建设了
我们还有个硬性规定：禁止跨层取数。DWD必须从ODS取数，不能直接连源数据库。这样数据血缘很清晰，出了问题也容易排查。

**DWD层的事实表有哪几种？**

事务事实表：一行代表一个具体的业务操作，通常只插入，不更新，是最常用的
周期快照事实表：在固定时间间隔对业务状态进行快照，一行代表某个时间点（天/周/月）的汇总状态，示例：每日库存快照表、每月账户余额表、每天订单状态汇总表
累积快照事实表：跟踪业务实体的完整生命周期，记录关键里程碑，示例：订单生命周期表（包含下单时间、支付时间、发货时间、签收时间等）
无事实的事实表：只有维度外键，没有度量值，示例：用户签到表、活动参与记录表

**你的DWS层是怎么设计的？有什么原则？**

DWS层我们定位得很明确——它是数仓的指标统一出口，下游只能从这一层拿指标。
设计原则有几个：
第一，面向主题域建表，不面向报表建表。 比如说我们建"用户行为日汇总表"，而不是"XX报表数据表"。因为报表会变，但主题是相对稳定的。
第二，派生指标有四个要素：原子指标、修饰词、时间周期和聚合维度。 比如"近7日北京地区用户下单数"——"下单数"是原子指标，"北京地区"是修饰词，"近7日"是时间周期，"用户"是聚合维度。这四个要素组合起来才是一个完整的派生指标。
第三，不同时间粒度分表存储。 日表和月表分开建，不会混在一张表里。这样查询的时候分区裁剪更清晰，性能也更好。
第四，适度冗余。 有些高频筛选的维度我们会直接冗余到DWS表里，避免频繁JOIN。当然这个要权衡，不能为了冗余而冗余。
我们有个硬性规定：DWD层不做跨过程指标拼接。比如订单和售后的关联分析，必须在DWS层做，DWD层只负责把各自的事实表建好。

**维度建模中，什么是事实表的"粒度"？为什么它很重要？**

粒度是事实表中一行数据所代表的业务含义，简单说就是**"一行数据代表什么"**。
比如说订单事实表，如果粒度是"每个订单项"，那么一个订单如果有3个商品，就会有3行记录。如果粒度是"每个订单"，那不管有多少商品，都只有1行。
为什么重要？ 因为粒度决定了：
第一，数据能不能下钻分析。 如果粒度是"订单级别"，你就没法分析单个商品的销售情况了。反之，如果粒度是"订单项级别"，你可以向上聚合到订单级别，但反过来不行。所以我们的原则是DWD层尽量保持最细粒度，汇总的事交给DWS层做。
第二，指标计算是否正确。 比如计算GMV，如果粒度搞错了，可能会重复计算。我们之前就踩过坑，有个表粒度是"用户-商品-天"，但计算销售额的时候直接SUM，结果同一个订单被算了多次。
第三，存储和性能的权衡。 粒度越细，数据量越大，但灵活性越高。我们DWD层用细粒度，DWS层根据分析需求做适当的汇总。

**数据仓库的模型设计方法有哪些？你更倾向于哪种？**

第一种是范式建模（ER模型）。 就是传统数据库的第三范式设计，消除数据冗余。优点是数据一致性好，存储高效。缺点是查询复杂，JOIN太多，性能差。这种在传统数仓 like Teradata用得比较多。
第二种是维度建模。 就是Fact+Dimension的星型模型，也是我主要用的方法。优点是查询性能好，业务理解门槛低。缺点是有一定冗余，但随着存储成本降低，这个已经不是大问题了。
一般主流都是用的维度建模，原因是
业务友好：星型模型很直观，业务人员容易理解性能优秀，减少了JOIN，查询效率高
生态成熟：工具链完善，从建模到可视化的支持都很好
当然，我们也不是完全不用范式。ODS层其实就比较接近范式模型，保持和源系统一致。到了DWD层才转为维度模型。实际项目中往往是混合使用，不是非黑即白。

**数仓中如何处理缓慢变化维？你们的实践是什么？**

缓慢变化维（SCD）是维度建模里的经典问题。比如用户的收货地址变了、商品的价格调整了，这些维度属性不是经常变，但变了之后怎么处理？
常见的处理方式：
SCD Type 1（覆盖更新）：直接UPDATE，不保留历史。适用于纠错场景，比如用户手机号录错了，直接改过来就行，不需要保留错误版本。
SCD Type 2（新增行）：每次变化插入新记录，用生效时间、失效时间和当前标识来管理。适用于需要完整历史追溯的场景，比如分析用户在不同城市的消费行为变化。
SCD Type 3（新增列）：加一个"上次值"字段，只能保留最近一次变化。用得比较少，因为扩展性差。
我们的实践：
对于DIM层，大部分场景用SCD Type 2。但如果用Iceberg，我们有更简单的方案——时间旅行。 直接查询某个时间点的快照，这样就不需要维护复杂的SCD Type 2结构了，大大简化了ETL逻辑。
选择建议：
如果用的是传统Hive/MySQL，SCD Type 2是标准做法
如果用Iceberg等数据湖技术，优先考虑时间旅行
关键是看业务是否需要追溯历史，不需要的话Type 1就够了

**数仓建设中，遇到过最大的挑战是什么？怎么解决的？**

最大的挑战应该是指标口径不一致导致的数据信任危机。
当时的情况是，公司高管看报表，发现财务部门的数据、运营部门的数据、数仓的数据对不上。同样是"月销售额"，三个数字都不一样。高管直接质疑："你们的数据到底哪个是对的？"
这个问题很严重，因为一旦失去信任，数仓的价值就没了。
根本原因：
各部门自己定义指标，没有统一管理
数仓早期为了快速响应需求，没有严格执行规范
源系统本身数据质量就有问题（比如退款没及时同步）
解决过程：
第一步：止血。 先盘点所有对外报表，标注每个指标的来源和计算逻辑，找出差异点。
第二步：统一口径。 拉着业务、财务、运营开会，逐一确认每个指标的官方定义。这个过程很痛苦，因为各方都有自己的"道理"，最后只能高管拍板。
第三步：技术重构。
建立指标字典，明确每个指标的归属层和计算逻辑
DWD层严格遵循"一个指标一张表"
DWS层作为唯一出口，下游只能用DWS的数据
废弃那些不规范的临时表
第四步：建立流程。
新增指标必须走评审流程
指标变更要通知所有下游
定期做数据对账
第五步：重建信任。
主动和业务团队对齐数据
建立数据质量报告，定期发布
有问题及时沟通，不藏着
经验教训：
规范必须严格执行，不能因为赶工期就妥协
指标管理要趁早，越到后面越难改
沟通比技术重要，很多问题不是技术解决不了，而是各方利益不一致
透明度很重要，数据来源、计算逻辑要公开，接受监督

## Trino部分
**你们为什么选 Trino？它跟 Spark SQL、Hive 这些引擎比起来，定位有什么不同？**

先说 Hive： Hive 是基于 MapReduce 的，延迟太高了，一个简单的 count 都要几十秒，而且只支持批处理，不适合交互式查询。我们现在基本只用 Hive 做元数据管理，查询已经不用了。
再说 Spark SQL： Spark SQL 其实也很强大，我们用它做批处理 ETL，跑 DWD、DWS 层的数仓任务。但它有个特点是批处理优先，每次查询要启动 Spark 应用，有初始化开销，而且资源调度比较重。
我再说说我自己做的这个项目，我要求时效性能做到每10分钟计算一次，第12分钟数据可见，这种是典型的高频upsert场景，trino的执行模式会更适合，首先，Spark启动开销太大太久，已经不符合时延要求了，我之前通过kyuubi复用了一个spark引擎解决了启动开销的问题，但是呢Spark不是为该场景设计的，你每个提交的SQL都会走重量级的优化器解析规划阶段，你多提交几个SQL就可能把Spark Driver干崩了，还没有细粒度的资源组控制，你不得不去加一些排队规划的逻辑，但是你即使加上了排队，但仍然可能遇到几个大的SQL把executor打崩，处理起来特别麻烦。
然后再说一下Trino，MPP架构的，纯内存计算，数据在内存中流转，不需要落地，所以延迟特别低，性能也别快，而且它是常驻引擎，因此先天就屏蔽了启动延迟问题，然后是连接器生态特别丰富，大部分数据它都能读，再者它对Iceberg的支持几乎可以覆盖我们目前所有的需求，即使不能覆盖，也可以通过Spark SQL补充。

**Trino 执行一条 SQL 的完整流程是怎样的？**

第一阶段，SQL解析，Coordinator 收到 SQL 后，先用 ANTLR 解析器把 SQL 字符串解析成 AST，这个阶段如果 SQL 语法有错误，会直接报错，比如少了个逗号、关键字拼错了。
第二阶段：语义分析，这一步会做几件事：元数据校验，类型推导，权限校验，如果表不存在或者权限不足，这个阶段就会报错。
第三阶段：逻辑计划生成与优化，AST 会被转换成逻辑执行计划，然后经过一系列优化规则，规则大概有谓词下推，列裁剪，常量折叠，Limit 下推等优化
第四阶段：物理计划阶段：将逻辑计划转换为具体的执行计划，包括任务分配、数据分区策略等执行细节。
最后就是任务执行了。

**能详细说说 Trino 的 Connector 架构吗？Iceberg Connector 是怎么工作的？**

Connector 是 Trino 最核心的扩展机制，Trino 本身不存储数据，所有数据读写都通过 Connector，Connector 架构分三层，第一层是Trino 定义的一套标准接口，包括ConnectorMetadata，用于提供表的元数据（Schema、列、分区等），ConnectorSplitManager，负责数据分片（Split），决定怎么并行读取，ConnectorPageSourceProvider，提供数据读取器，返回 Page，ConnectorPageSinkProvider，提供数据写入器（INSERT 时用），所有 Connector 都要实现这些接口，这样 Trino 才能统一管理，第二层是具体 Connector 实现，比如 Iceberg Connector、MySQL Connector、Hive Connector，每个都实现上面的 SPI。比如 Iceberg 表、MySQL 数据库、HDFS 文件等。我以 Iceberg Connector 为例，说一下查询流程：
第一步：元数据加载
当你查询 SELECT * FROM iceberg.catalog.table 时，Trino 会通过 Iceberg Connector 的 Metadata 模块去读取 Iceberg 的元数据。
Iceberg 的元数据是分层的：
Metadata JSON：记录表的 Schema、分区、快照（Snapshot）信息
Manifest List：记录每次快照的 Manifest 文件列表
Manifest File：记录每个数据文件的元数据（分区、行数、列的 min/max 等）
Data File：实际的 Parquet/ORC 数据文件
Connector 会先读 Metadata JSON，找到当前快照，然后读 Manifest List，再读 Manifest File，最后得到数据文件列表。
第二步：分区裁剪
如果 SQL 有 WHERE 条件，比如 WHERE biz_date = '2024-01-01'，Connector 会利用 Manifest 里的分区信息，过滤掉不满足条件的分区，只读取相关的数据文件。
第三步：文件过滤
即使分区确定了，可能还有很多文件。Connector 会利用 Manifest 里的列统计信息（min/max、null count）过滤掉不包含目标数据的文件。
比如 WHERE user_id = 12345，如果某个文件的 user_id 范围是1至100，那这个文件就可以跳过。
第四步：读取数据
确定要读哪些文件后，SplitManager 会为每个文件（或者文件的一部分）创建一个 Split，分发给 Worker 并行读取。
Worker 通过 PageSourceProvider 创建 PageSource，读取 Parquet 文件，转换成 Trino 的 Page 格式返回。
Parquet 读取还有两个优化：
列裁剪：只读查询需要的列，Parquet 是列式存储，这个效果很明显
谓词下推：Parquet 的页级别统计信息可以过滤掉不满足条件的数据页
第五步：INSERT 写入流程（如果有的话）
当我们用 Trino 执行 INSERT INTO iceberg.table ... 时：
Connector 创建 PageSink
Worker 将数据写入临时文件（Parquet 格式）
写入完成后，更新 Manifest 和 Metadata JSON，创建新快照
注意：这里会触发 Iceberg 的小文件问题，所以我们定期跑 Compaction 合并小文件

**Trino 是内存计算，那它的内存是怎么管理的？跟 Spark 比有什么不同？**

Trino的内存模型从宏观上来看分两类，第一类是堆内存，第二类是堆内存预留空间，用于GC和系统开销，再说说堆内存，堆内存又分为用户内存，系统内存，以及堆预留空间，和Spark不一样的是Spark中将堆预留空间又分为存储内存和执行内存，并可以动态分配，Trino没有这么细，但是Trino可以严格限制每个查询可以使用的内存，超出限制内存会直接kill掉查询，Spark的话会溢写到磁盘或者直接报OOM，因为Trino少了动态分配因此也会对比Spark的内存分配速度要快一点。

**Trino 查询慢，你一般怎么排查和优化？**

第一步：我会先跑 EXPLAIN ANALYZE，看执行计划和实际统计信息，重点看几个指标：Input vs Output 比例：如果 TableScan 读了 10 亿行，最后只输出 1 万行，说明过滤不够早，耗时最长的 Operator：可能是 Join、Aggregate 或者数据倾斜，数据分布：看每个 Task 处理的数据量是否均匀
第二步：针对性优化，根据瓶颈不同，优化策略也不同：如果是TableScan 慢（读数据慢），可能原因：没做分区裁剪、没做谓词下推、文件太小（小文件问题）、文件格式不好（比如 CSV vs Parquet），分区裁剪和谓词下推就要检查where条件写得对不对，小文件问题的话就要检查Iceberg表的维护策略，然后是Join慢的话，就得看是不是大表 Join 大表，Shuffle 数据量大，或者数据倾斜，数据倾斜的话就得尝试过滤或者加盐，场景三的就是Exchange 慢，exchange是发生在网络中的，看看是不是Stage 之间传输数据量太大，太大的话就可以尝试一下减少中间结果集。

**生产环境肯定要考虑安全，Trino 的权限控制你们是怎么做的？**

安全这块我确实花了不少精力，但说实话我在这块做的并不深入，因为安全认证一般分为认证和鉴权以及传输安全，第一个步骤是要开启https加密，说实话，这个步骤我也了解得不多，我当初部署的时候几乎依赖于AI才勉强配置成功了，是通过k8s的cert-manager发放的凭证，等cert-manager创建了证书secrets以后，挂载到Trino中并在配置中启动后就可以了，然后是认证，我在k8s中部署了一个LLDAP服务，Trino和kyuubi公用这个lldap服务，但实际上一些ldap的高级特性我没有用，我目前的用法是通过lldap保存用户名和密码，再然后是鉴权，说实话我Trino的鉴权根本没做，但是Iceberg底层数据我是通过Rest Catalog做了用户只能访问哪些目录，库，和表的。